In [20]:
import os
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torch.distributed as dist
import torch.multiprocessing as mp
from  utils import *

os.chdir("./utils") # replace with your path
from builder import *
from dataloader import *
from densenet import *
from embedding_extractor_test import *
from preprocessing import *
from utils import *
from parse import *
import densenet as models

from builder import *
from parse import *

In [21]:
# 加载数据
class MoCoDataset_test(torch.utils.data.Dataset):
    def __init__(self, filepath, transform = None):
        self.path = filepath
        self.transform = transform

    def __len__(self):
        self.data = np.load(self.path,allow_pickle=True)
        return self.data['x'].shape[0]   
        
    def __getitem__(self,i):
        
        x = self.data['x'][i]
        y = self.data['y'][i]
        return x, y
    
def load_para(save_path):
    checkpoint = torch.load(save_path)
    #载入模型参数
    state_dict = checkpoint['state_dict']
    new_state_dict = {}
    for k, v in state_dict.items():
        name = k[7:]  # remove 'module.' of dataparallel
        if name.endswith(".weight") or name.endswith(".bias"):
            if name.split(".")[2] in ["0","2"]:
                name = f'{name.split(".")[0]}.{name.split(".")[1]}.{name.split(".")[3]}'
        new_state_dict[name] = v
    return new_state_dict

def model_out(test_dataset,model,labe_data):
    X = []
    Y = []
    for i in range(len(test_dataset)):
        data,label= test_dataset[i]
        q = torch.tensor(data,dtype=torch.float32)
        q = q.unsqueeze(0)
        #frature = q.detach().numpy()[0]
        frature = model.encoder_q(q).detach().numpy()[0]
        if label in list(labe_data["Patient_ID"].values):
            label = labe_data[labe_data["Patient_ID"] == label]["Cancer"].values[0]
            if label in ["UNCLASSIFIED","nan"]:
                continue
            X.append(frature)
            Y.append(label)
    X = np.array(X)
    Y = np.array(Y)
    return X,Y

In [30]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import umap

# 【关键设置】确保导出的PDF在AI中文字可编辑，不碎掉
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

def self_umap(X, Y, pdf_dir=None):
    unique_labels = np.unique(Y)
    
    # ---------------------------------------------------------
    # 论文级专业色板（合并了 Tableau 和 ColorBrewer 的稳重色系）
    # ---------------------------------------------------------
    # 这些颜色饱和度适中，辨识度高，看起来更“高级”
    color_list_hex = [
        '#1F77B4', '#FF7F0E', '#2CA02C', '#D62728', '#9467BD', # Tableau 10
        '#8C564B', '#E377C2', '#7F7F7F', '#BCBD22', '#17BECF', 
        '#4E79A7', '#F28E2B', '#E15759', '#76B7B2', '#59A14F', # Tableau 20 subsets
        '#EDC948', '#B07AA1', '#FF9DA7', '#9C755F', '#BAB0AC',
        '#377EB8', '#4DAF4A', '#984EA3', '#FF7F00', '#FFFF33', # Brewer Set1
        '#A65628', '#F781BF', '#999999', '#66C2A5', '#FC8D62', # Brewer Set2
        '#8DA0CB', '#E78AC3', '#A6D854', '#FFD92F', '#E5C494', '#B3B3B3'
    ]

    # 如果分类超过色板数量，自动通过循环或采样补充
    if len(unique_labels) > len(color_list_hex):
        # 使用 hls 空间生成更多颜色，保持视觉亮度一致
        import colorsys
        n = len(unique_labels)
        extra_colors = []
        for i in range(n - len(color_list_hex)):
            # 生成不那么刺眼的颜色
            rgb = colorsys.hls_to_rgb(i/n, 0.6, 0.7) 
            extra_colors.append('#%02x%02x%02x' % tuple([int(x*255) for x in rgb]))
        color_list_hex.extend(extra_colors)

    # ---------------------------------------------------------
    
    # UMAP 模型初始化
    umap_model = umap.UMAP(n_neighbors=10, min_dist=0.1, n_components=2, random_state=2023)
    umap_result = umap_model.fit_transform(X)

    plt.figure(figsize=(4, 3), dpi=300) # 建议固定画布大小
    
    for i, tumor_type in enumerate(unique_labels):
        mask = Y == tumor_type
        # s=5 控制点的大小，alpha=0.7 增加重叠感，edgecolors='none' 让点看起来更柔和
        plt.scatter(umap_result[mask, 0], umap_result[mask, 1], 
                    c=color_list_hex[i], 
                    label=tumor_type,
                    s=16, 
                    alpha=0.75, 
                    edgecolors='none')

    # 修饰图表样式，更符合论文审美
    ax = plt.gca()
    # 隐藏上方和右方的边框，更显清爽
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # 调整图例：使用 smaller 字体，并放在外面
    plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), 
               title="Tumor Types", frameon=False, fontsize=8)
    
    plt.title('UMAP (MoCo Features)', fontsize=12, fontweight='bold', pad=15)
    plt.xlabel('UMAP 1', fontsize=10)
    plt.ylabel('UMAP 2', fontsize=10)

    if pdf_dir is None:
        plt.show()
    else:
        # 使用 bbox_inches='tight' 确保图例不被裁剪
        plt.savefig(f"{pdf_dir}", bbox_inches='tight', dpi=300)
        plt.close()

In [24]:
args = moco_parser().parse_args('')
#args
args = moco_parser().parse_args('')
model_dict = models.__dict__

kwargs = {"randomzero" : 0.3}


In [25]:
'''
    GDSC
'''
#载入数据
test_dataset = MoCoDataset_test("./data/zscore_test_Depmap_ProteinCodingGene_mRNA.npz")
train_dataset = MoCoDataset_test("./data/zscore_train_Depmap_ProteinCodingGene_mRNA.npz")
# 读取标签数据
label_dir = "./data/Depmap/Model.csv"
labe_data = pd.read_csv(label_dir)
labe_data["Patient_ID"] = labe_data["ModelID"]
labe_data["Cancer"] = labe_data["OncotreeLineage"]


In [26]:
len(labe_data["OncotreeLineage"].unique().tolist())

35

In [27]:
def load_para_new(save_path):
    # 在这里指定 map_location
    checkpoint = torch.load(save_path, map_location='cpu') 
    
    state_dict = checkpoint['state_dict']
    new_state_dict = {}
    for k, v in state_dict.items():
        name = k[7:] # remove 'module.'
        # ... 保持你原有的逻辑不变 ...
        if name.endswith(".weight") or name.endswith(".bias"):
            if name.split(".")[2] in ["0","2"]:
                name = f'{name.split(".")[0]}.{name.split(".")[1]}.{name.split(".")[3]}'
        new_state_dict[name] = v
    return new_state_dict

In [28]:
with torch.no_grad():
    model = MoCo(
            model_dict["densenet27"],
            num_batches = 32, 
            in_features = args.in_features, 
            dim=args.moco_dim, 
            K=args.moco_k, 
            m=args.moco_m, 
            T=args.moco_t, 
            mlp=args.mlp,
            **kwargs)
    # 1. 确保模型本身在 CPU 上
    model.cpu() 
    save_path = "./result/mRNA_protein/checkpoint_0199.pth.tar"
    # 2. 直接加载处理好的参数（此时 load_para 内部已经处理了 CPU 映射）
    model.load_state_dict(load_para(save_path))

    # 3. 开启评估模式
    model.eval()

/tmp/ipykernel_94120/3042842697.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(save_path)


In [29]:
# 原始数据-测试集
X_train_before = []
Y_train_before = []
for i in range(len(test_dataset)):
    data,label= test_dataset[i]
    if label in list(labe_data["Patient_ID"].values):
            label = labe_data[labe_data["Patient_ID"] == label]["Cancer"].values[0]
            if label in ["UNCLASSIFIED","nan"]:
                continue
            X_train_before.append(data)
            Y_train_before.append(label)
    
X_train_before = np.array(X)
Y_train_before = np.array(Y)

# 原始数据，训练集
X_test_before = []
Y_test_before = []
for i in range(len(train_dataset)):
    data,label= train_dataset[i]
    if label in list(labe_data["Patient_ID"].values):
            label = labe_data[labe_data["Patient_ID"] == label]["Cancer"].values[0]
            if label in ["UNCLASSIFIED","nan"]:
                continue
            X_test_before.append(data)
            Y_test_before.append(label)


X_test_before = np.array(X_test_before)
Y_test_before = np.array(Y_test_before)

#测试集
X_test_after,Y_test_after = model_out(test_dataset,model,labe_data)
#训练集
X_train_after,Y_train_after = model_out(train_dataset,model,labe_data)

In [31]:
self_umap(X_train_before, Y_train_before,pdf_dir="./原始数据-测试集_performance_Depmap_ProteinCodingGene.pdf")

In [32]:
self_umap(X_test_before, Y_test_before,pdf_dir="./原始数据-训练集_performance_Depmap_ProteinCodingGene.pdf")

In [33]:
self_umap(X_test_after, Y_test_after,pdf_dir="./降维数据-测试集_performance_Depmap_ProteinCodingGene.pdf")

In [34]:
self_umap(X_train_after, Y_train_after,pdf_dir="./降维数据-训练集_performance_Depmap_ProteinCodingGene.pdf")